# AI Resume Analyzer using Google Gemini

In [ ]:
from PyPDF2 import PdfReader
from google import genai
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# GEMINI CONFIGURATION
# ============================================================
GEMINI_API_KEY = "Replace Your API Key Here"

# Initialize Gemini Client
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# ============================================================
# PDF TO TEXT
# ============================================================
def pdf_to_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

# Extract text from local PDF resume
pdf_path = "resume.pdf"  # Replace with your PDF filename
try:
    resume_text = pdf_to_text(pdf_path)
    print(f"Extracted {len(resume_text)} characters from resume.")
except Exception as e:
    print(f"Error reading PDF file (make sure '{pdf_path}' exists in the current directory): {e}")

In [ ]:
# ============================================================
# GEMINI REQUEST
# ============================================================
def ask_gemini(resume_text, instruction):
    max_characters = 30000
    if len(resume_text) > max_characters:
        resume_text = resume_text[:max_characters] + "\n\n[Resume text was shortened]"

    prompt = f"""
You are an expert AI Resume Analyzer and professional career advisor.
Analyze ONLY the resume provided below.

IMPORTANT RULES:
1. Do not invent information.
2. Do not invent skills.
3. If something is missing, clearly say "Not mentioned".
4. Give practical recommendations.
5. Use headings and bullet points.

============================================================
RESUME
============================================================
{{resume_text}}

============================================================
TASK
============================================================
{{instruction}}
"""
    
    # Match the models used in app.py
    models = ["gemini-3.6-flash", "gemini-3.5-flash-lite"]
    last_error = None
    for model in models:
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt.format(resume_text=resume_text, instruction=instruction)
            )
            if response and response.text:
                return response.text
        except Exception as e:
            last_error = e
            continue
    
    raise RuntimeError(f"Gemini could not generate a response. Last error: {last_error}")

In [ ]:
# ============================================================
# RUN ANALYSIS
# ============================================================

# 1. Summary
summary_prompt = """
Create a detailed professional summary of this resume.
Include Candidate Profile, Education, Technical Skills, Programming Languages, AI/ML Skills, Projects, Certifications, Experience, Achievements, Key Strengths, and Career Direction.
"""
try:
    summary_result = ask_gemini(resume_text, summary_prompt)
    print("--- SUMMARY ---\n")
    print(summary_result)
except Exception as e:
    print(f"Error running Summary analysis: {e}")

In [ ]:
# 2. Strengths
strength_prompt = """
Analyze the strengths of this resume.
Consider technical skills, projects, certifications, and career potential, explaining why each is valuable.
"""
try:
    strength_result = ask_gemini(resume_text, strength_prompt)
    print("--- STRENGTHS ---\n")
    print(strength_result)
except Exception as e:
    print(f"Error running Strengths analysis: {e}")

In [ ]:
# 3. Weaknesses
weakness_prompt = """
Analyze the weaknesses of this resume.
Check missing skills, weak sections, project descriptions, ATS friendliness, formatting, and create a prioritized improvement plan.
"""
try:
    weakness_result = ask_gemini(resume_text, weakness_prompt)
    print("--- WEAKNESSES & SUGGESTIONS ---\n")
    print(weakness_result)
except Exception as e:
    print(f"Error running Weaknesses analysis: {e}")

In [ ]:
# 4. Job Titles
job_title_prompt = """
Suggest suitable job roles for this candidate, ranked from most to least suitable.
Include suitability percentage match, strengths, and skills gaps for each role.
"""
try:
    job_title_result = ask_gemini(resume_text, job_title_prompt)
    print("--- RECOMMENDED JOB TITLES ---\n")
    print(job_title_result)
except Exception as e:
    print(f"Error running Job Titles analysis: {e}")

In [ ]:
# 5. ATS Score
ats_prompt = """
Act as an ATS-style resume evaluation system.
Provide an estimated ATS score out of 100 and identify weak sections, missing keywords, and suggestions to improve.
"""
try:
    ats_result = ask_gemini(resume_text, ats_prompt)
    print("--- ATS SCORE ---\n")
    print(ats_result)
except Exception as e:
    print(f"Error running ATS analysis: {e}")

In [ ]:
# 6. Skills
skills_prompt = """
Extract and organize the skills actually present in the resume into categories.
Provide verified list, strongest skills, and areas to improve.
"""
try:
    skills_result = ask_gemini(resume_text, skills_prompt)
    print("--- SKILLS ANALYSIS ---\n")
    print(skills_result)
except Exception as e:
    print(f"Error running Skills analysis: {e}")